# 02 · Validation results without pitch-rate scaling
DDQN ($\kappa_t = 1$, nominal rate 0.4 °/s) versus ROSCO on the five validation scenarios.

* Table 4 (`tab:ddqn_combined_results`).
* Check of the upper power-deviation threshold (0.1 MW) used by the adaptive rule (Section 5.5).

Analysis window: 200–5200 s.

In [1]:
import pandas as pd
from utils import load_fast_file, window, scaled_std, CHANNELS, P_RATED

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

# scenario -> (ROSCO file, DDQN file without scaling)
SCENARIOS = {
    "8 m/s sawtooth":  ("val_saw08_rosco.out",  "val_saw08_ddqn_0.4dps.out"),
    "10 m/s sawtooth": ("val_saw10_rosco.out",  "val_saw10_ddqn_0.4dps.out"),
    "15 m/s sawtooth": ("val_saw15_rosco.out",  "val_saw15_ddqn_0.4dps.out"),
    "11 m/s TI=10%":   ("val_11ti10_rosco.out", "val_11ti10_ddqn_0.4dps.out"),
    "15 m/s TI=10%":   ("val_15ti10_rosco.out", "val_15ti10_ddqn_0.4dps.out"),
}
data = {name: (load_fast_file(ro)[0], load_fast_file(rl)[0]) for name, (ro, rl) in SCENARIOS.items()}

## Table 4: mean and scaled STD

In [2]:
UNITS = {"GenPwr": "[kW]", "TwrBsMyt": "[kN·m]", "RootMyb1": "[kN·m]"}

results = {}
for name, (rosco, ddqn) in data.items():
    ro, rl = window(rosco), window(ddqn)
    col = {}
    for ch in CHANNELS:
        col[f"{ch} mean DDQN {UNITS[ch]}"] = rl[ch].mean()
        col[f"{ch} mean ROSCO {UNITS[ch]}"] = ro[ch].mean()
        col[f"{ch} scaled STD DDQN"] = scaled_std(rl[ch], ro[ch])
        col[f"{ch} scaled STD ROSCO"] = scaled_std(ro[ch], ro[ch])
    col["Last time step DDQN [s]"] = ddqn["Time"].iloc[-1]
    results[name] = col

table4 = pd.DataFrame(results).astype(float).round(3)
table4

,8 m/s sawtooth,10 m/s sawtooth,15 m/s sawtooth,11 m/s TI=10%,15 m/s TI=10%
GenPwr mean DDQN [kW],1826.982,3372.573,3375.123,3370.575,3378.631
GenPwr mean ROSCO [kW],1826.983,3229.896,3369.731,3347.209,3370.621
GenPwr scaled STD DDQN,0.349,0.247,0.128,0.084,0.077
GenPwr scaled STD ROSCO,0.349,0.367,0.344,0.105,0.137
TwrBsMyt mean DDQN [kN·m],43903.562,55208.576,29542.446,45808.552,30316.449
TwrBsMyt mean ROSCO [kN·m],43903.571,51507.237,29408.855,45099.925,30183.345
TwrBsMyt scaled STD DDQN,0.296,0.890,0.339,0.184,0.143
TwrBsMyt scaled STD ROSCO,0.296,0.269,0.288,0.122,0.119
RootMyb1 mean DDQN [kN·m],6419.605,7974.292,3999.824,6479.518,3911.416
RootMyb1 mean ROSCO [kN·m],6419.605,7492.026,3980.567,6384.686,3888.210


## Upper power-deviation threshold
The upper threshold of the adaptive rule (0.1 MW) is compared with the power STD of the final training simulation and of the 11 m/s validation scenario.

In [3]:
THRESHOLD = 100.0  # kW

# Final training run: Δβ = 0.010°, ε0 = 0.025, ε_decay = 10^-3.25 (final controller in the paper)
FINAL_TRAINING_RUN = "train_11ti10_ddqn_db0.010_pr800_eps0.025_dec3.25.out"

runs = {
    "Training, DDQN final": load_fast_file(FINAL_TRAINING_RUN)[0],
    "Training, ROSCO": load_fast_file("train_11ti10_rosco.out")[0],
    "Validation 11 m/s, DDQN": data["11 m/s TI=10%"][1],
    "Validation 11 m/s, ROSCO": data["11 m/s TI=10%"][0],
}
rows = []
for name, df in runs.items():
    for t0, t1 in [(200, 2000), (200, 5200)]:
        p = window(df, t0, t1)["GenPwr"]
        rows.append({"Simulation": name, "Window [s]": f"{t0}-{t1}", "Mean [kW]": p.mean(),
                     "STD [kW]": p.std(), "STD / threshold": p.std() / THRESHOLD})
print(f"Threshold = {THRESHOLD / 1000:.3f} MW = {100 * THRESHOLD / P_RATED:.2f} % of P_rated")
pd.DataFrame(rows).round(2)

Threshold = 0.100 MW = 2.97 % of P_rated


,Simulation,Window [s],Mean [kW],STD [kW],STD / threshold
0,"Training, DDQN final",200-2000,3369.24,76.89,0.77
1,"Training, DDQN final",200-5200,3361.93,105.23,1.05
2,"Training, ROSCO",200-2000,3339.41,125.06,1.25
3,"Training, ROSCO",200-5200,3340.53,134.30,1.34
4,"Validation 11 m/s, DDQN",200-2000,3358.79,96.78,0.97
5,"Validation 11 m/s, DDQN",200-5200,3370.57,92.40,0.92
6,"Validation 11 m/s, ROSCO",200-2000,3350.16,104.77,1.05
7,"Validation 11 m/s, ROSCO",200-5200,3347.21,115.65,1.16
